In [ ]:
!pip install torch torchvision opencv-python numpy matplotlib pandas tqdm onnx

In [ ]:
import os
import cv2
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
CONFIG = {
    "IMG_SIZE": 256,
    "BATCH_SIZE": 32,
    "LEARNING_RATE": 1e-4,
    "EPOCHS": 50,
    "NUM_KEYPOINTS": 21,
    "NUM_CLASSES": 1, # Hand
    "BACKBONE_DIM": 64,
    "WEIGHT_DECAY": 1e-4
}

In [ ]:
class HandPoseDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=256, augment=True):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.img_size = img_size
        self.augment = augment
        self.img_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        label_path = os.path.join(self.label_dir, self.img_files[idx].replace('.jpg', '.txt'))

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        labels = np.loadtxt(label_path).reshape(-1, 5 + CONFIG['NUM_KEYPOINTS']*2)

        if self.augment:
            img, labels = self.apply_augmentation(img, labels)

        img = transforms.ToTensor()(img)
        img = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img)

        return img, torch.FloatTensor(labels)

In [ ]:
 def apply_augmentation(self, img, labels):
        # Random Horizontal Flip
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
            labels[:, 1] = 1.0 - labels[:, 1] # Flip bbox x
            for i in range(5, 5 + CONFIG['NUM_KEYPOINTS']*2, 2):
                labels[:, i] = 1.0 - labels[:, i] # Flip keypoint x

        # Random Brightness
        if random.random() < 0.5:
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
            hsv[:,:,2] = hsv[:,:,2] * random.uniform(0.7, 1.3)
            img = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

        return img, labels

In [ ]:
# Dummy paths for demonstration
train_dataset = HandPoseDataset('dataset/train/images', 'dataset/train/labels', augment=True)
val_dataset = HandPoseDataset('dataset/val/images', 'dataset/val/labels', augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=False,
    num_workers=4
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

In [ ]:
class SiLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, kernel_size//2, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

In [ ]:
class SPPF(nn.Module):
    def __init__(self, in_channels, out_channels, k=5):
        super().__init__()
        c_ = in_channels // 2
        self.cv1 = ConvBlock(in_channels, c_, 1, 1)
        self.cv2 = ConvBlock(c_ * 4, out_channels, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k//2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))

In [ ]:
class YOLOHandPose(nn.Module):
    def __init__(self, num_keypoints=21):
        super().__init__()
        self.num_keypoints = num_keypoints

        # Simplified Backbone
        self.backbone = nn.Sequential(
            ConvBlock(3, 16, 3, 2),    # 128x128
            ConvBlock(16, 32, 3, 2),   # 64x64
            ConvBlock(32, 64, 3, 2),   # 32x32
            ConvBlock(64, 128, 3, 2),  # 16x16
            SPPF(128, 128)
        )

        # Neck
        self.neck = nn.Sequential(
            ConvBlock(128, 64, 1, 1),
            ConvBlock(64, 64, 3, 1),
            ConvBlock(64, 32, 1, 1)
        )

        # Head: 4 (bbox) + 1 (obj) + 42 (21 keypoints * 2)
        out_dim = 4 + 1 + (num_keypoints * 2)
        self.head = nn.Conv2d(32, out_dim, 1, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.neck(x)
        return self.head(x).permute(0, 2, 3, 1) # B, H, W, C

In [ ]:
class WingLoss(nn.Module):
    def __init__(self, omega=10, epsilon=2):
        super().__init__()
        self.omega = omega
        self.epsilon = epsilon

    def forward(self, pred, target):
        delta = torch.abs(pred - target)
        # Small errors (linear)
        loss1 = self.omega * torch.log(1 + delta / self.epsilon)
        # Large errors (constant)
        loss2 = delta - C
        C = self.omega - self.omega * math.log(1 + self.omega / self.epsilon)
        loss = torch.where(delta < self.omega, loss1, loss2)
        return loss.mean()

In [ ]:
def ciou_loss(pred_boxes, target_boxes):
    # pred_boxes & target_boxes format: [x, y, w, h]
    px, py, pw, ph = pred_boxes[..., 0], pred_boxes[..., 1], pred_boxes[..., 2], pred_boxes[..., 3]
    tx, ty, tw, th = target_boxes[..., 0], target_boxes[..., 1], target_boxes[..., 2], target_boxes[..., 3]

    # Intersection area
    inter = (torch.min(px, tx) * torch.min(py, ty)).clamp(0)
    union = pw * ph + tw * th - inter
    iou = inter / union.clamp(min=1e-6)

    # Center distance
    center_dist = (px - tx)**2 + (py - ty)**2

    # Diagonal distance
    diag_dist = pw**2 + ph**2 + tw**2 + th**2

    # Aspect ratio penalty
    v = (4 / math.pi**2) * (torch.atan(tw / th.clamp(min=1e-6)) - torch.atan(pw / ph.clamp(min=1e-6)))**2
    alpha = v / (1 - iou + v + 1e-6).clamp(min=1e-6)

    ciou = iou - center_dist / diag_dist.clamp(min=1e-6) - alpha * v
    return 1 - ciou.mean()

In [ ]:
model = YOLOHandPose(num_keypoints=CONFIG['NUM_KEYPOINTS']).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LEARNING_RATE'], weight_decay=CONFIG['WEIGHT_DECAY'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'])

bbox_loss_fn = ciou_loss
kp_loss_fn = WingLoss()
obj_loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
for epoch in range(CONFIG['EPOCHS']):
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}")
    for imgs, targets in pbar:
        imgs = imgs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        preds = model(imgs) # B, H, W, C

        # Simplified loss calculation for demonstration
        # Assuming targets are aligned to grid cells (requires target assignment logic not shown here)
        pred_bbox = preds[..., :4]
        pred_obj = preds[..., 4]
        pred_kp = preds[..., 5:]

        tgt_bbox = targets[..., 1:5]
        tgt_obj = torch.ones_like(pred_obj)
        tgt_kp = targets[..., 5:]

        loss_bbox = bbox_loss_fn(pred_bbox, tgt_bbox)
        loss_obj = obj_loss_fn(pred_obj, tgt_obj)
        loss_kp = kp_loss_fn(pred_kp, tgt_kp)

        loss = loss_bbox + loss_obj + 5.0 * loss_kp # Weight keypoints higher

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item(), bbox=loss_bbox.item(), kp=loss_kp.item())

    scheduler.step()
    print(f"Epoch {epoch+1} Avg Loss: {total_loss / len(train_loader):.4f}")

    # Validation step would go here...

In [ ]:
dummy_input = torch.randn(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']).to(device)
torch.onnx.export(
    model,
    dummy_input,
    "yolov26_hand_pose.onnx",
    export_params=True,
    opset_version=12,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print("Model successfully exported to ONNX!")